# 프로젝트 1 - Weekend 1: API와 체인 기반 FAQ 시스템
| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 |
| **소요 시간** | 5시간 (30분 x 10사이클) |
| **핵심 기술** | Python, OpenAI API, LangChain LCEL, Gradio |

## 환경 설정

In [4]:
!pip install -q openai langchain-openai python-dotenv gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.6 MB/s eta 0:00:00


In [5]:
# colab 사용 시 아래 주석 해제
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [1]:
# (로컬) 환경 설정
# import os
# from dotenv import load_dotenv
# load_dotenv()

# from openai import OpenAI
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser

# client = OpenAI()
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# print("✅ 환경 설정 완료")

## 📦 실습용 샘플 데이터

In [13]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

📦 FAQ 데이터 로드 완료: 10개 QA, 5개 테스트 질의


---
## 사이클 1: 첫 API 호출

OpenAI API로 주택청약 관련 질문을 보내고 답변을 받아보세요. `system` 역할에 "주택청약 전문 상담원"을 설정하세요.

In [11]:
# 사이클 1
"""[계획]

어떻게 해야할까?

1. 어떤 라이브러리 호출하고, 어떤 클래스를 객체로 삼아야 할지 알아봐야함.
2. invoke는 바로 사용하진 않고, message를 구성한 다음, 역할 2개를 각각 2개의 딕셔너리로 구성했던 것 같음
  - {system, content}
  - {user(human), content}
2-1. prompt를 구성할 때, "system", "human" 등, 튜플로 하는 방식이 있고,SystemMessage 등으로 사용하는 방식이 있는데 뭐가 다른건지 모르겠음.
    (from langchain_core.messages import SystemMessage, HumanMessage, AIMessage)

3. 우선은 랭체인 방식으로, llm, prompt, parser 3단계를 구성한 다음 '|' 를 이용하여 chain 객체를 만들고, chain.invoke로 질문을 전송 및 응답을 얻음.
"""
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model = "gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 최대한 정확한 답변을 주세요."),
    ("user", "{question}")
])

parser = StrOutputParser()

chain = prompt | llm | parser
role = "주택청약 전문가 및 부동산 관련 법률 담당하는 공무원"
result = chain.invoke({"role": role, "question": "청약통장 가입하려면 어떻게 해요?"})
print(result)



청약통장을 가입하려면 아래의 단계를 따르시면 됩니다.

1. **청약통장 종류 선택**: 청약통장은 주택청약종합저축과 청약예금, 청약부금 등 여러 종류가 있습니다. 주택청약종합저축이 가장 일반적입니다. 원하는 주택청약 통장의 종류를 선택합니다.

2. **신청서 작성**: 은행이나 상업은행의 은행지점을 방문하여 청약통장 가입신청서를 작성합니다. 또는 인터넷뱅킹을 통해 온라인으로 신청할 수도 있습니다.

3. **신분증 지참**: 가입 시 신분증(주민등록증, 운전면허증 등)을 지참해야 합니다.

4. **첫 납입금 납부**: 가입 시 최초 납입금이 필요합니다. 통장마다 최소 가입금액이 다르므로 해당 금융 기관에서 확인하고 납부합니다.

5. **가입 완료**: 필요한 서류와 납입금이 처리되면 청약통장이 발급됩니다. 이 통장을 통해 주택청약 시 필요한 자금을 적립할 수 있습니다.

가입 시 유의할 점은, 청약통장은 1인 1계좌로 제한되며, 가입 후 일정 기간 이상 유지해야 한다는 점입니다. 추가로, 청약통장은 주택청약 시 가점이 부여되므로, 최대한 빨리 가입하시는 것이 좋습니다.


---
## 사이클 2: FAQ 데이터 탐색

`SAMPLE_FAQ_DATA`에서 카테고리별 FAQ 개수를 세고, `difficulty`가 `"easy"`인 항목만 필터링해서 출력하세요.

In [29]:
# 사이클 2
"""
단순히 카테고리별 개수를, 변수(딕셔너리)에서 세는 태스크인듯.
1. 카테고리별 FAQ 개수 어떻게 세는지?
"""
# 1. 카테고리별 FAQ 개수
# 1.1. set()을 활용 (총 카테고리 개수) == 틀렸지만, 해봄

category_set = set()

for faq_item in SAMPLE_FAQ_DATA:
  print(faq_item["category"])
  category_set.add(faq_item["category"]) # 모두 set에 때려넣고
num_category = len(category_set)
print(num_category)

청약통장
청약통장
청약자격
청약자격
특별공급
특별공급
일반공급
당첨/계약
당첨/계약
기타
6


In [27]:
# 1. 카테고리별 FAQ 개수
# 1.2. 카운터 활용(있으면 1 증가)
category_counts = {}

for faq_item in SAMPLE_FAQ_DATA:
    category = faq_item["category"]
    # 딕셔너리에 해당 카테고리가 있으면 +1, 없으면 1로 초기화
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1
print("카테고리별 FAQ 개수:", category_counts)

카테고리별 FAQ 개수: {'청약통장': 2, '청약자격': 2, '특별공급': 2, '일반공급': 1, '당첨/계약': 2, '기타': 1}


In [33]:
# 2. difficulty가 "easy"인 항목만 필터링해서 출력.

for faq_item in SAMPLE_FAQ_DATA:
  if faq_item['difficulty'] == 'easy':
    print(faq_item)

{'id': 'FAQ001', 'category': '청약통장', 'question': '주택청약종합저축이란 무엇인가요?', 'answer': '주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨', 'keywords': ['청약종합저축', '만능통장', '납입', '가입'], 'difficulty': 'easy'}
{'id': 'FAQ005', 'category': '청약자격', 'question': '주택 청약 신청 자격 조건은 무엇인가요?', 'answer': '1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능', 'keywords': ['청약자격', '만19세', '무주택', '세대주'], 'difficulty': 'easy'}
{'id': 'FAQ017', 'category': '당첨/계약', 'question': '당첨자 발표는 어떻게 확인하나요?', 'answer': '1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인', 'keywords': ['당첨자발표', '청약홈', 'SMS알림', '서류제출'], 'difficulty': 'easy'}
{'id': 'FAQ023', 'category': '기타', 'question': '청약홈 사이트는 어떻게 이용하나요?', 'answer': '청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공', 'keywords': 

---
## 사이클 3: FAQ 검색 함수

질문 문자열을 받아 키워드 매칭으로 관련 FAQ를 찾는 `search_faq(query, faq_data, top_k=3)` 함수를 만들고, `SAMPLE_TEST_QUERIES`로 테스트하세요.

### seqrch_faq() 구현 (단순 키워드 매칭 로직)

In [40]:
keywords = [faq_item["keywords"] for faq_item in SAMPLE_FAQ_DATA]
keywords

[['청약종합저축', '만능통장', '납입', '가입'],
 ['1순위', '가입기간', '예치금', '투기과열지구'],
 ['청약자격', '만19세', '무주택', '세대주'],
 ['무주택', '세대원', '소형주택', '분양권'],
 ['특별공급', '기관추천', '다자녀', '신혼부부', '생애최초'],
 ['신혼부부', '혼인기간', '소득기준', '가점'],
 ['가점제', '추첨제', '84점', '투기과열지구'],
 ['당첨자발표', '청약홈', 'SMS알림', '서류제출'],
 ['재당첨제한', '10년', '7년', '세대원'],
 ['청약홈', '공인인증서', '간편인증', '가점계산']]

In [52]:
# 사이클 3

"""[계획]
질문 문자열을 받아야함.
- 질문은 questionn 변수에 넣고, prompt에 전달해 주면 될 것같음.
- search_faq() 함수는 query, faq_data, top_k = 3을 입력받음.
  - query는 뭘까? prompt는 시스템과 유저 content로 구성되어 있는데, 우선 그냥 question=query라고 생각하자.
  - faq_data는 그냥 SAMPLE_FAQ_DATA
  - top_k는 어디서 쓸까?
    => 우선 top_p는 LLM이 생성한 토큰 후보들의 확률이 특정 확률이 되는 것 까지만 후보군으로 취합?
    => top_k는 특정 개수(k) 까지만 토큰 후보를 선택한다는 말이었던 것 같음.
    = top_k... question 문자열에서 중요한 단어(chunk?)만 추출하도록 구성하면 될까?
    = 그건 아닌 것 같고, 검색 결과로 찾은 FAQ 항목 개수(k=3)를 의미하는 것 같음. 토큰 예측 후보군 중 k개를 뽑는것과 상관 X
    = top_k = 3을, 그냥 '답변에서 추출한 **키워드 개수**'로 고려하여, 기존과 매칭 등을 활용해보려함.

faq_data 중, k=3개를 뽑아오도록 해야함.
- 우선 질문 내용에 SAMPLE_FAQ_DATA를 넣어주긴 해야하고, 어떤 식으로 제공해야할까? => 문자열 list로 만든 후 prompt에 합산?

"""
# question = "무주택자인데, 청약 신청 절차 및 당첨 이후 작업에 대해 알려주세요."
faq_data = SAMPLE_FAQ_DATA

def search_faq(query: str, faq_data: dict, top_k:int =3):

  # context_str = ""
  # for faq in SAMPLE_FAQ_DATA:
  #     context_str += f"ID: {faq['id']}\n질문: {faq['question']}\n답변: {faq['answer']}\n키워드: {', '.join(faq['keywords'])}\n\n"

  # ============ 1단계: 사용자 질문에서 키워드 추출 ============

  # 1) 입력받은 SAMPLE_FAQ_DATA에서 키워드 추출
  extracted_keywords = set()

  for faq in faq_data:
    extracted_keywords.update(faq['keywords'])

  extracted_keywords_str = ", ".join(extracted_keywords)


  # 2) prompt에 SAMPLE_FAQ_DATA의 키워드 정보 주입
  llm = ChatOpenAI(model = "gpt-4o-mini")
  prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 {role}입니다. \n
          [참고 키워드 목록]: {extracted_keywords_str} \n
          - 사용자의 질문에서 가장 중요한 핵심 명사 키워드 {top_k}개를 추출해주세요.\n
          - 최대한 키워드 목록에서 뽑아주세요.\n
          - 반드시 쉼표(,)로만 구분해서 출력하세요. (예시: 청약, 통장, 조건)"""
         ),
        ("user", "{question}")
    ])

  parser = StrOutputParser()

  keyword_chain = prompt | llm | parser # 체인 구성

  role = "주택청약 전문가 및 부동산 관련 법률 담당하는 공무원"

  # 3) 체인 실행
  keyword_extracted = keyword_chain.invoke({
    'role': role,
    'top_k': top_k,
    'question': query,
    'extracted_keywords_str': extracted_keywords_str
  })

  # [중요] LLM이 준 결과는 하나의 긴 문자열("청약, 통장, 조건")이므로,
  # 쉼표(,)를 기준으로 잘라서 파이썬 리스트(['청약', '통장', '조건'])로 만들어야 함.
  parsed_keywords =[kw.strip() for kw in keyword_extracted.split(",")]

  # ============ 2단계: 단순 키워드 매칭으로 관련 FAQ 찾기 ============
  """
  키워드 매칭은 어떻게 하지? 그냥 단순한 키워드 매칭.
  match_score = {}로 두고,
    - id
    - match_score
  for문으로 DATA를 순회하여 `keyword_extracted`결과와 단순히 매칭되면.. 함?

  return은 가장 매칭이 높은 id를 가져와 출력
  """

  # 1) 키워드 매칭 점수 부여
  match_score = {}
  for faq in faq_data:
    match_score[faq['id']] = 0 # 각 FAQ의 점수를 0으로 초기화

    for keyword in parsed_keywords: # 질문에서 추출한 키워드를 순회하며 스코어링
      if keyword in faq['keywords']: # 추출된 키워드가 해당 FAQ 목록에 있으면 1점 추가
        match_score[faq['id']] += 1

  # 2) 점수가 가장 높은 id의 FAQ를 출력
  # 딕셔너리를 점수(value) 기준으로 내림차순 정렬 (예:[('FAQ001', 3), ('FAQ003', 1), ('FAQ002', 0)])
  sorted_scores = sorted(match_score.items(), key=lambda item: item[1], reverse=True)

  # 3) 가장 점수가 높은(1등) FAQ의 ID와 점수를 가져옴
  best_id = sorted_scores[0][0]
  best_score = sorted_scores[0][1]

  # 4) 전체 데이터(faq_data)에서 1등 ID와 일치하는 전체 FAQ 정보를 찾음
  best_faq_data = None
  for faq in faq_data:
    if faq['id'] == best_id:
      best_faq_data = faq
      break

  return {
      "extracted_keywords": parsed_keywords,
      "best_score": best_score,
      "best_faq": best_faq_data
  }

# --- 실행 테스트 ---
query = "청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?"
result = search_faq(query, SAMPLE_FAQ_DATA, top_k=3)

result


{'extracted_keywords': ['당첨자발표', '서류제출', '분양권'],
 'best_score': 2,
 'best_faq': {'id': 'FAQ017',
  'category': '당첨/계약',
  'question': '당첨자 발표는 어떻게 확인하나요?',
  'answer': '1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인',
  'keywords': ['당첨자발표', '청약홈', 'SMS알림', '서류제출'],
  'difficulty': 'easy'}}

In [35]:
# 기타 코드

context_str = ""
for faq in SAMPLE_FAQ_DATA:
    context_str += f"ID: {faq['id']}\n질문: {faq['question']}\n답변: {faq['answer']}\n키워드: {', '.join(faq['keywords'])}\n\n"
context_str

'ID: FAQ001\n질문: 주택청약종합저축이란 무엇인가요?\n답변: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨\n키워드: 청약종합저축, 만능통장, 납입, 가입\n\nID: FAQ004\n질문: 청약통장 1순위 조건은 무엇인가요?\n답변: 1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입\n키워드: 1순위, 가입기간, 예치금, 투기과열지구\n\nID: FAQ005\n질문: 주택 청약 신청 자격 조건은 무엇인가요?\n답변: 1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능\n키워드: 청약자격, 만19세, 무주택, 세대주\n\nID: FAQ006\n질문: 무주택자 기준은 무엇인가요?\n답변: 본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함\n키워드: 무주택, 세대원, 소형주택, 분양권\n\nID: FAQ009\n질문: 특별공급의 종류에는 어떤 것이 있나요?\n답변: 1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대\n키워드: 특별공급, 기관추천, 다자녀, 신혼부부, 생애최초\n\nID: FAQ010\n질문: 신혼부부 특별공급 조건은 무엇인가요?\n답변: 1) 혼인기간

### search_faq() 함수 구현2: (예정)

---
## 사이클 4: 검색 결과 + LLM 답변 생성

검색된 FAQ를 system prompt에 넣어 답변을 생성하는 `ask_faq(question, faq_data, client)` 함수를 만드세요. 답변과 함께 참고한 FAQ 목록도 반환하세요.

In [62]:
# 사이클 4
# 검색된 FAQ를 system prompt에 넣어 답변을 생성
"""[계획]
- search_faq로 뽑은, 가장 관련성이 높은 FAQ 항목의 context를 추출
- 추출한 내용을 {} 항목으로 제공 및 체인(chain) 구성

"""
#result = search_faq(query, SAMPLE_FAQ_DATA, top_k=3) # top_k는 키워드 추출 개수를 의미함

# 함수 구현

def ask_faq(question, faq_data, client):
  searched_faq = search_faq(query, SAMPLE_FAQ_DATA, top_k=3)
  keywords = searched_faq['extracted_keywords']
  related_faq = searched_faq['best_faq']

  related_context = f"""
          카테고리: {related_faq['category']}\n
          질문: {related_faq['question']}\n
          답변: {related_faq['answer']}\n
          키워드: {related_faq['keywords']}
  """
  llm = ChatOpenAI(model = "gpt-4o-mini")
  role = "주택청약 전문가 및 부동산 관련 법률 담당하는 공무원"
  prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 {role}입니다. \n
          [참고 FAQ]: {related_context} \n
          - 사용자의 질문에 대한 답변 시, '[참고 FAQ]'의 내용을 참고하여 답변하세요.\n
        """
         ),
        ("user", "{question}")
    ])

  parser = StrOutputParser()

  keyword_chain = prompt | llm | parser # 체인 구성

  role = "주택청약 전문가 및 부동산 관련 법률 담당하는 공무원"

  answer = keyword_chain.invoke({
    'role': role,
    'question': question,
    'related_context': related_context
    })

  return answer, related_faq

# === 실행 ===
query = "청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?"
client = OpenAI()

answer = ask_faq(query, SAMPLE_FAQ_DATA, client)

print(answer[0])
print("관련 FAQ 항목:", answer[1])

청약에 당첨된 이후에는 다음과 같은 절차를 따라야 합니다:

1. **서류 제출 기간 확인**: 당첨된 후에는 정해진 기간 내에 필요한 서류를 제출해야 하므로, 이 기간을 반드시 확인하세요.
   
2. **계약 일정 확인**: 당첨된 아파트의 계약 일정도 확인하시기 바랍니다. 계약 날짜에 맞춰 준비를 해야 합니다.

3. **서류 준비**: 필요한 서류를 미리 준비해 두시고, 누락되지 않도록 주의하세요.

4. **청약홈 확인**: 추가적으로 궁금한 사항이 있다면 청약홈(www.applyhome.co.kr)에 접속하여 관련 정보를 확인할 수 있습니다.

또한, **SMS 알림 서비스**를 신청하면 당첨자 발표 및 계약 관련 정보를 문자로 받을 수 있으니 활용해 보세요.
관련 FAQ 항목: {'id': 'FAQ017', 'category': '당첨/계약', 'question': '당첨자 발표는 어떻게 확인하나요?', 'answer': '1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인', 'keywords': ['당첨자발표', '청약홈', 'SMS알림', '서류제출'], 'difficulty': 'easy'}


In [ ]:
# Dummy
"""
best_faq': {'id': 'FAQ017',
  'category': '당첨/계약',
  'question': '당첨자 발표는 어떻게 확인하나요?',
  'answer': '1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인',
  'keywords': ['당첨자발표', '청약홈', 'SMS알림', '서류제출'],

   f"ID: {faq['id']}\n질문: {faq['question']}\n답변: {faq['answer']}\n키워드: {', '.join(faq['keywords'])}\n\n"
"""

---
## 사이클 5: PromptTemplate

`ChatPromptTemplate`으로 `{context}`와 `{question}` 변수를 사용하는 FAQ 답변용 프롬프트를 만들고, 카테고리 분류용 프롬프트도 하나 더 만들어서 각각 테스트하세요.

In [68]:
# 사이클 5
from langchain_core.prompts import ChatPromptTemplate

# ============ 1. FAQ 답변용 프롬프트 ============

def make_context(query, SAMPLE_FAQ_DATA):
  searched_faq = search_faq(query, SAMPLE_FAQ_DATA, top_k=3)
  keywords = searched_faq['extracted_keywords']
  related_faq = searched_faq['best_faq']

  related_context = f"""
          카테고리: {related_faq['category']}\n
          질문: {related_faq['question']}\n
          답변: {related_faq['answer']}\n
          키워드: {related_faq['keywords']}
  """
  return related_context

prompt_faq = ChatPromptTemplate.from_messages([
        ("system", """
        유저의 질문에 대해서 {related_context}를 이용하여 답변하세요.

        """),
        ("user", "{question}")
    ])


query = "청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?"
related_context = make_context(query, SAMPLE_FAQ_DATA)
parser = StrOutputParser()

faq_answer_chain = prompt_faq | llm | parser

faq_answer = faq_answer_chain.invoke({
    "related_context": related_context,
    "question": query
})

print("1. FAQ 답변용 프롬프트", '\n', faq_answer, '\n')

# ============ 2. 카테고리 분류용 프롬프트 ============

def extract_categories_context(SAMPLE_FAQ_DATA):
  category_set = set()

  for faq_item in SAMPLE_FAQ_DATA:
    category_set.add(faq_item["category"]) # 모두 set에 때려넣고

  category_context = ", ".join(category_set) # 쉼표로 구분된 하나의 문자열로 변환

  return category_context

prompt_category = ChatPromptTemplate.from_messages([
        ("system", """
        - [카테고리 목록]: {category_context}
        - 유저의 질문에 대해 어떤 카테고리에 속하는지 '[카테고리 목록]'을 참고하여 하나의 카테고리로 분류하세요.
        """),
        ("user", "{question}")
    ])

llm = ChatOpenAI(model = "gpt-4o-mini")
parser = StrOutputParser()

category_chain = prompt_category | llm | parser

query = "청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?"
category_classification = category_chain.invoke({
    "category_context": extract_categories_context(SAMPLE_FAQ_DATA),
    "question": query
})

print("2. 질문 카테고리 분류:",'\n',category_classification,'\n')

1. FAQ 답변용 프롬프트 
 당첨에 성공한 후에는 다음과 같은 절차를 따르셔야 합니다:

1) **서류 제출**: 당첨 후 필요한 서류를 준비하여 제출해야 합니다. 정확한 서류 목록은 당첨자 발표 시 확인할 수 있으니 반드시 체크하세요.

2) **계약 일정 확인**: 계약을 체결하는 일정이 따로 정해져 있으니, 이를 확인하여 스마트하게 준비하세요.

3) **청약홈 활용**: '청약홈(www.applyhome.co.kr)' 에 들어가셔서 당첨자 조회 메뉴를 클릭하여 당첨 여부 및 서류 제출 기간을 확인할 수 있습니다.

4) **SMS 알림 서비스**: 당첨 및 서류 제출 관련 정보는 문자 알림으로도 받을 수 있으니, 문자 알림 서비스 신청을 고려해보세요.

이러한 절차를 잘 확인하고 이행하시면 당첨 후 문제가 없을 것입니다. 

2. 질문 카테고리 분류: 
 당첨/계약 



---
## 사이클 6: LCEL 체인

`prompt | llm | StrOutputParser()` 패턴으로 FAQ 답변 체인(`faq_chain`)을 만들고, 질문 2개로 테스트하세요. `.stream()`으로 스트리밍 출력도 해보세요.

In [75]:
# 사이클 6
from langchain_core.output_parsers import StrOutputParser

def make_context(query, SAMPLE_FAQ_DATA):
  searched_faq = search_faq(query, SAMPLE_FAQ_DATA, top_k=3)
  keywords = searched_faq['extracted_keywords']
  related_faq = searched_faq['best_faq']

  related_context = f"""
          카테고리: {related_faq['category']}\n
          질문: {related_faq['question']}\n
          답변: {related_faq['answer']}\n
          키워드: {related_faq['keywords']}
  """
  return related_context

context = make_context(query, SAMPLE_FAQ_DATA)

llm = ChatOpenAI(model = "gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
        ("system", "{context}를 참조하여 답변하세요."),
        ("user", "{question}")
    ])

parser = StrOutputParser()

chain = prompt | llm | parser

query1 = "청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?"

result1 = chain.invoke({
    "context": make_context(query1, SAMPLE_FAQ_DATA),
    "question": query1
})


query2 = "서류 제출 단계에서 어떤 서류를 요구하는지 알 수 있을까?"

result2 = chain.invoke({
    "context": make_context(query2, SAMPLE_FAQ_DATA),
    "question": query2
})

# invoke 사용
# print("질문1:",result1)
# print("===================="*5)
# print("질문2:", result2)

# stream 사용 ===> query1에 대한 답변만 나옴
# for chunk in chain.stream({
#         "context": make_context(query1, SAMPLE_FAQ_DATA),
#         "question": query1
#     },
#                               {
#         "context": make_context(query2, SAMPLE_FAQ_DATA),
#         "question": query2
#     }):
#   print(chunk, end="", flush=True)


# stream 사용 (질문 2개 모두 출력)
  # 리스트로 묶어서 반복 실행
queries = [query1, query2]

for q in queries:
    print(f"\n[답변 시작: {q}]\n")
    for chunk in chain.stream({"context": make_context(q, SAMPLE_FAQ_DATA), "question": q}):
        print(chunk, end="", flush=True)
    print("\n" + "="*30)


[답변 시작: 청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?]

청약에 당첨된 이후에는 다음과 같은 절차를 따라야 합니다:

1. **당첨자 발표 확인**: 청약홈(www.applyhome.co.kr)에 접속하여 당첨자 조회 메뉴를 클릭하여 자신의 당첨 여부를 확인합니다.

2. **서류 제출 준비**: 당첨 후 요구되는 서류를 준비합니다. 이 서류는 각 단지나 기관에 따라 다를 수 있으니 반드시 확인해야 합니다.

3. **서류 제출 기간 확인**: 당첨 이후 서류 제출 기간을 확인하고, 기간 내에 필요한 서류를 모두 제출합니다.

4. **계약 일정 확인**: 계약 일정도 반드시 확인하여 계약 체결을 위한 준비를 합니다.

5. **SMS 알림 서비스 신청**: 청약홈에서 SMS 알림 서비스 신청이 가능합니다. 이를 통해 중요한 일정이나 변경 사항에 대한 알림을 받을 수 있습니다.

이 과정을 통해 성공적으로 계약을 진행할 수 있습니다. 필요한 모든 서류와 일정을 미리 체크하여 원활한 진행을 하는 것이 중요합니다.

[답변 시작: 서류 제출 단계에서 어떤 서류를 요구하는지 알 수 있을까?]

주택 청약 신청 시 서류 제출 단계에서 요구되는 서류는 다음과 같습니다:

1) **청약통장 가입 증명서**: 청약통장 가입 여부를 증명하는 서류.
2) **신분증**: 주민등록증 또는 운전면허증 등 본인을 확인할 수 있는 신분증.
3) **세대주 또는 세대원 증명서**: 주민등록등본 등 세대 구성원 관계를 증명할 수 있는 서류.
4) **무주택 확인서**: 무주택 상태를 증명하기 위한 서류 (예: 무주택세대구성원확인서).
5) **소득 증명서류**: 소득을 증명할 수 있는 서류 (예: 소득금액증명원 등, 해당 경우).

각 주택 유형이나 지역에 따라 추가 서류가 필요할 수 있으므로, 구체적인 청약 조건에 맞춰 확인하는 것이 중요합니다.


---
## 사이클 7: 검색

질문을 넣으면 자동으로 FAQ 검색 → 답변 생성하는 `rag_chain`을 만드세요. `SAMPLE_TEST_QUERIES` 5개로 테스트하세요.

In [77]:
# 사이클 7
# 질문을 넣으면 자동으로 FAQ 검색 → 답변 생성하는 rag_chain


from langchain_core.runnables import RunnablePassthrough

# 1. RAG 체인 구성
# ⭐ RunnablePassthrough는 이름 그대로 입력받은 데이터를 '그대로 다음 단계로 통과'시키는 역할
# 질문이 들어오면 (retrieval_step을 통해 context 생성) + (질문 그대로 통과) -> 프롬프트 -> LLM -> 파서
rag_chain = (
    {"context": lambda x: make_context(x, SAMPLE_FAQ_DATA), "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

# 2. 테스트 실행

SAMPLE_TEST_QUERIES =[
   "청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?",
   "서류 제출 단계에서 어떤 서류를 요구하는지 알 수 있을까?",
   "무주택 세대주인데 추가 서류는 필요하니?",
   "소득 제한 요건은 있을까?",
   "평균 소요되는 서류단계 처리 기간은?"

]
for test in SAMPLE_TEST_QUERIES:
    query = test
    print(f"\n[질문]: {query}")
    response = rag_chain.invoke(query)
    print(f"[답변]: {response}")
    print("-"*50)


[질문]: 청약에 당첨된 이후에 내가 어떤 일을 해야 할지 알려줄래?
[답변]: 청약에 당첨된 이후에는 다음과 같은 절차를 따라야 합니다:

1. **당첨자 발표 확인**: 청약홈(www.applyhome.co.kr)에 접속하여 당첨자 조회 메뉴를 클릭하여 본인의 당첨 여부를 확인하세요.

2. **서류 준비**: 당첨이 확인되면, 서류 제출을 위한 필요한 서류를 준비해야 합니다. 구체적인 서류 목록은 청약홈이나 해당 주택청약 관련 기관에서 확인할 수 있습니다.

3. **서류 제출**: 각 주택청약에 따라 서류 제출 기간이 정해져 있으니, 그 기간에 맞추어 서류를 제출해야 합니다.

4. **계약 일정 확인**: 서류 제출 후에는 계약 일정도 확인해야 합니다. 계약 일정에 맞춰 계약을 체결해야 하며, 해당 내용은 보통 당첨자 발표와 함께 고지됩니다.

5. **SMS 알림 서비스 신청**: 향후 정보를 쉽게 받기 위해 SMS 알림 서비스를 신청할 수 있습니다.

이 모든 과정은 반드시 기간 안에 진행해야 하니, 주의 깊게 확인하시기 바랍니다.
--------------------------------------------------

[질문]: 서류 제출 단계에서 어떤 서류를 요구하는지 알 수 있을까?
[답변]: 주택 청약 신청 시 서류 제출 단계에서 요구되는 서류는 다음과 같습니다:

1. **청약신청서**: 청약을 신청하기 위한 기본 서류입니다.
2. **청약통장 가입증명서**: 청약통장에 대한 가입 사실을 증명하는 서류입니다.
3. **신분증 사본**: 본인 확인을 위한 주민등록증, 운전면허증 등 신분증 사본이 필요합니다.
4. **주택거주지 증명서**: 현재 거주하는 주택의 정보를 확인할 수 있는 서류 (예: 주민등록등본)입니다.
5. **소득증명서**: 소득을 증명할 수 있는 서류 (예: 근로소득원천징수영수증, 사업소득신고서 등)입니다.
6. **가족관계증명서**: 세대구성원을 입증하기 위한 가족관계증명서도 필요할 수 있습니다.

추가 요구 서류는

---
## 사이클 8: 에러 처리

빈 입력, 500자 초과, 숫자만 입력 등을 검증하고 `try/except`로 API 오류를 처리하는 `safe_ask(question, rag_chain)` 함수를 만드세요. 정상/에러 케이스 6가지 이상 테스트하세요.

In [84]:
# 사이클 8
"""[계획]
빈입력, 500자 초과, 숫자만 입력 등을 검증하는 함수(safe_ask) 구현

그냥 함수 내부에서 if문 등으로 question을 검증.
"""

import time

# context를 이용하여 답변하는 rag_chain.
rag_chain = (
    {"context": lambda x: make_context(x, SAMPLE_FAQ_DATA), "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

def safe_ask(question, rag_chain):
    """
    사용자의 입력(question)을 검증하고 API 실행을 안전하게 처리하는 함수.
    """
    try:
        cleaned_question = question.strip() # 앞뒤 공백 제거
        # 1. 빈 문자열 검증
        if not cleaned_question:
            raise ValueError("질문이 입력되지 않았습니다. 내용을 입력해주세요.")

        # 2. 500자 초과 검증
        if len(cleaned_question) > 500:
            raise ValueError(f"질문이 너무 깁니다. 500자 이내로 입력해주세요. (현재 {len(cleaned_question)}자)")

        # 3. 숫자만 입력된 경우 검증 (isdigit() 활용)
        if cleaned_question.isdigit():
            raise ValueError("숫자 데이터는 질문할 수 없습니다.")

        return rag_chain.invoke(cleaned_question)

    except ValueError as e:
        return f"[입력 오류] {str(e)}"

    except Exception as e:
        return f"[시스템 오류] 답변을 생성하는 중 문제가 발생했습니다. 잠시 후 다시 시도해주세요. (상세: {str(e)})"

# 1. 빈 문자열 검증
query1 = ""
ans1 = safe_ask(query1, rag_chain)
print(ans1)

# 2. 500자 초과 검증
query2 = "검"*505
ans2 = safe_ask(query2, rag_chain)
print(ans2)

# 3. 숫자만 입력된 경우 검증
query3 = "12345"
ans3 = safe_ask(query3, rag_chain)
print(ans3)

# 4. 정상 질문
query4 = "무주택자인데 청약 신청 절차가 어떻게 되나요?"
ans4 = safe_ask(query4, rag_chain)
print(ans4)

# 5. 공백(스페이스, 엔터, 탭 등)만 입력된 경우 검증
query5 = "   \n \t  "
ans5 = safe_ask(query5, rag_chain)
print(f"5. 공백만 입력: {ans5}")

# 6. 문자열이 아닌 잘못된 데이터 타입(None, 정수 등)이 입력된 경우
query6 = None
ans6 = safe_ask(query6, rag_chain)
print(f"6. 잘못된 타입(None) 입력: {ans6}")


[입력 오류] 질문이 입력되지 않았습니다. 내용을 입력해주세요.
[입력 오류] 질문이 너무 깁니다. 500자 이내로 입력해주세요. (현재 505자)
[입력 오류] 숫자 데이터는 질문할 수 없습니다.
무주택자로서 청약 신청 절차는 다음과 같습니다:

1. **청약통장 개설**: 청약 신청을 위해 먼저 청약통장을 개설해야 합니다. 청약통장은 은행이나 저축은행 등에서 가입할 수 있습니다.

2. **조건 확인**: 자신이 원하는 주택의 청약자격 조건(예: 무주택 세대구성원)과 청약 유형(국민주택 또는 민영주택)에 따라 자격을 확인합니다.

3. **청약 공고 확인**: 주택청약 홈페이지나 관련 기관의 공고를 통해 청약 일정 및 신청 가능한 주택 정보를 확인합니다.

4. **신청서 작성**: 청약공고에 맞춰 신청서를 작성합니다. 필요한 서류(주민등록등본, 청약통장 가입증명서 등)를 준비합니다.

5. **청약 신청**: 정해진 기간 내에 관련 기관에 청약 신청을 합니다. 인터넷 청약 또는 창구에서 신청할 수 있습니다.

6. **추첨 결과 확인**: 청약 신청 후 진행된 추첨 결과를 확인하고, 당첨 시 계약 체결 및 잔금 납부 절차를 진행합니다.

이와 같은 절차를 따르면 무주택자로서 청약 신청을 할 수 있습니다.
5. 공백만 입력: [입력 오류] 질문이 입력되지 않았습니다. 내용을 입력해주세요.
6. 잘못된 타입(None) 입력: [시스템 오류] 답변을 생성하는 중 문제가 발생했습니다. 잠시 후 다시 시도해주세요. (상세: 'NoneType' object has no attribute 'strip')


---
## 사이클 9: Gradio 채팅 UI

`gr.ChatInterface`로 지금까지 만든 RAG 체인을 웹 채팅 UI로 만드세요. 제목, 설명, 예시 질문 5개를 설정하세요.

In [87]:
# 사이클 9
"""[계획, 정보]
Gradio 컴포넌트 선택 기준
- 단순 함수 (입력 → 출력)    → gr.Interface
- 챗봇 UI                    → gr.ChatInterface  (type="messages" 필수)
- 복잡한 레이아웃             → gr.Blocks (Row/Column 자유 배치)
"""


import gradio as gr

# ====== 1. Gradio 연동용 함수 작성 ======
  # gr.ChatInterface의 fn 인자에 넣기 위한 함수
  # fn 함수는 반드시 (message, history) → str 시그니처를 가져야 함
def chat_with_bot(message, history):
    """
    - message: 현재 사용자 입력 (str)
    - history: 이전 대화 목록 (list of dict)
    """
    response = safe_ask(message, rag_chain)
    return response


# ====== 2. UI 화면 텍스트 설정 ======
# 제목
chat_title = "주택청약 AI 챗봇"

# 설명
chat_description = """
주택청약 종합저축, 가입 조건, 당첨 후 절차 등 부동산 청약에 대해 궁금한 점을 물어보세요 (※ 최대 500자 입력 가능)
"""

# 예시
chat_examples =[
    "주택청약종합저축이란 무엇인가요?",
    "무주택 세대주 요건이 어떻게 되나요?",
    "청약 당첨 이후 절차는 어떻게 진행되나요?",
    "당첨자 발표 확인하고 서류 제출하는 법 알려줘",
    "청약통장 매월 얼마씩 납입하는게 가장 유리할까요?"
]

# ====== 3. ChatInterface 생성 및 실행 ======
demo = gr.ChatInterface(
    fn=chat_with_bot,             # 함수 연결
    title=chat_title,             # 챗봇 제목
    description=chat_description, # 챗봇 설명
    examples=chat_examples,       # 예시 질문 리스트
)

# 실행 (share=True로 외부 접속 링크 생성)
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://182e5fec3e2b76fdec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 사이클 10: 최종 통합 테스트

전체 파이프라인(입력 검증 → 검색 → 답변 생성)을 하나의 함수로 정리하고, 10개 질문으로 테스트하세요. 각 질문의 응답 시간, 참고 FAQ 수를 포함한 결과표를 출력하세요. Gradio UI도 최종 버전으로 만드세요.

#### 함수 통합

In [88]:
# 사이클 10
"""[계획]

1. 전체 파이프라인을 하나의 함수로 통합.

"""
import gradio as gr
import time

def query_pipeline(question, data):
  """
  사용자의 질문을 받아 검증, FAQ 검색, 최종 답변 생성까지 한 번에 수행하는 파이프라인
  """

  # 공통으로 사용할 LLM과 Parser 정의
  llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
  parser = StrOutputParser()

  try:
      # ====== [1단계] 입력 검증 ======
      if not isinstance(question, str):
          raise ValueError("질문은 문자열 형태로 입력해야 합니다.")

      cleaned_question = question.strip()

      if not cleaned_question:
          raise ValueError("질문이 입력되지 않았습니다. 내용을 입력해주세요.")

      if len(cleaned_question) > 500:
          raise ValueError(f"질문이 너무 깁니다. 500자 이내로 입력해주세요. (현재 {len(cleaned_question)}자)")

      if cleaned_question.isdigit():
          raise ValueError("숫자 데이터는 질문할 수 없습니다. 문장으로 질문해주세요.")


      # ====== [2단계] 기존 데이터 검색 ======

      # 1) 전체 데이터에서 키워드 풀(Pool) 추출
      extracted_keywords = set()
      for faq in data:
          extracted_keywords.update(faq.get('keywords',[]))
      extracted_keywords_str = ", ".join(extracted_keywords)

      # 2) LLM을 이용해 질문에서 키워드 추출
      keyword_prompt = ChatPromptTemplate.from_messages([
          ("system", """당신은 주택청약 전문가 및 부동산 관련 법률 담당하는 공무원입니다.[참고 키워드 목록]: {extracted_keywords_str}

          - 사용자의 질문에서 가장 중요한 핵심 명사 키워드 {top_k}개를 추출해주세요.
          - 최대한 위[참고 키워드 목록]에서 단어를 뽑아주세요.
          - 반드시 쉼표(,)로만 구분해서 출력하세요. (예시: 청약, 통장, 조건)"""),
          ("user", "{question}")
      ])

      keyword_chain = keyword_prompt | llm | parser

      keyword_extracted = keyword_chain.invoke({
          'top_k': top_k,
          'question': cleaned_question,
          'extracted_keywords_str': extracted_keywords_str
      })

      # 추출된 키워드 문자열을 리스트로 변환
      parsed_keywords =[kw.strip() for kw in keyword_extracted.split(",")]

      # 3) 단순 키워드 매칭으로 가장 관련성 높은 FAQ 찾기
      match_score = {}
      for faq in data:
          match_score[faq['id']] = 0
          for keyword in parsed_keywords:
              if keyword in faq.get('keywords',[]):
                  match_score[faq['id']] += 1

      # 점수 기준 내림차순 정렬 및 관련도 1순위 FAQ 데이터 찾기
      sorted_scores = sorted(match_score.items(), key=lambda item: item[1], reverse=True)
      best_id = sorted_scores[0][0]

      best_faq_data = None
      for faq in data:
          if faq['id'] == best_id:
              best_faq_data = faq
              break

      # ====== [3단계] 답변(Chain) 생성 (RAG) ======
      # 1) 검색된 관련도 1순위 FAQ 데이터를 바탕으로 프롬프트에 넣을 Context 생성
      if best_faq_data:
          related_context = (
              f"카테고리: {best_faq_data['category']}\n"
              f"질문: {best_faq_data['question']}\n"
              f"답변: {best_faq_data['answer']}\n"
              f"키워드: {', '.join(best_faq_data['keywords'])}"
          )
      else:
          related_context = "관련된 FAQ를 찾을 수 없습니다."

      # 2) 최종 답변 생성 프롬프트
      qa_prompt = ChatPromptTemplate.from_messages([
          ("system", """당신은 친절하고 전문적인 주택청약 안내 챗봇입니다.
          아래 제공된 [관련 FAQ 정보]를 바탕으로 사용자의 질문에 답변해주세요.
          제공된 정보에 없는 내용이라면 임의로 지어내지 말고, "해당 내용은 FAQ에 존재하지 않습니다"라고 안내하세요.

          [관련 FAQ 정보]
          {context}"""),
          ("user", "{question}")
      ])

      qa_chain = qa_prompt | llm | parser

      # 3) 최종 결과 생성
      final_answer = qa_chain.invoke({
          "context": related_context,
          "question": cleaned_question
      })

      return final_answer

  # 에러 exception
  except ValueError as e:
      # 1단계에서 걸러진 사용자의 잘못된 입력 처리
      return f"[입력 오류] {str(e)}"

  except Exception as e:
      # LLM 연결 문제, 파이썬 내부 에러 등 예상치 못한 시스템 오류 처리
      return f"[시스템 오류] 답변을 생성하는 중 문제가 발생했습니다. (상세: {str(e)})"


#### 질문 테스트

In [90]:
import time
import pandas as pd

# 테스트할 질문 리스트
test_queries = [
    "청약통장 가입하려면 어떻게 해요?",
    "1순위 되려면 뭐가 필요해요?",
    "신혼부부 특공 자격이 궁금해요",
    "가점이 높으면 유리한가요?",
    "당첨되면 어떻게 확인해요?",
    "12345",  # 숫자 입력 에러 케이스
    ""        # 빈 입력 에러 케이스
]

results = []

for q in test_queries:
    start_time = time.time()

    top_k = 3

    response = query_pipeline(q, SAMPLE_FAQ_DATA)
    end_time = time.time()

    results.append({
        "질문": q if q else "(공백)",
        "응답": response[:100] + "..." if len(response) > 100 else response,
        "소요시간(초)": f"{end_time - start_time:.2f}s"
    })

# 결과 출력
df_results = pd.DataFrame(results)
display(df_results)

,질문,응답,소요시간(초)
0,청약통장 가입하려면 어떻게 해요?,해당 내용은 FAQ에 존재하지 않습니다.,1.10s
1,1순위 되려면 뭐가 필요해요?,"1순위 조건은 주택 유형에 따라 다릅니다. \n\n1) 민영주택: 수도권 12개월,...",3.96s
2,신혼부부 특공 자격이 궁금해요,해당 내용은 FAQ에 존재하지 않습니다.,0.96s
3,가점이 높으면 유리한가요?,"네, 가점이 높으면 유리합니다. 신혼부부 특별공급의 경우, 혼인기간이 짧을수록 그리...",2.03s
4,당첨되면 어떻게 확인해요?,당첨자 발표를 확인하는 방법은 다음과 같습니다:\n\n1) 청약홈(www.apply...,1.63s
5,12345,[입력 오류] 숫자 데이터는 질문할 수 없습니다. 문장으로 질문해주세요.,0.00s
6,(공백),[입력 오류] 질문이 입력되지 않았습니다. 내용을 입력해주세요.,0.00s


#### Gradio

In [91]:

# 1. Gradio용 연결 함수
def chat_with_bot(message, history):
    # 앞에서 만든 파이프라인 함수를 호출하여 결과를 반환
    # (실제 환경에서는 앞서 작성한 query_pipeline 코드가 메모리에 있어야 합니다)
    return query_pipeline(message, SAMPLE_FAQ_DATA)

# 2. 멋진 테마 설정 (블루 톤의 부드럽고 전문적인 테마)
custom_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
    font=[gr.themes.GoogleFont("Pretendard"), "Arial", "sans-serif"]
)

# 3. 레이아웃을 마음대로 짤 수 있는 gr.Blocks 사용
with gr.Blocks(theme=custom_theme, title="주택청약 AI 도우미") as demo:

    # 상단 헤더 영역
    with gr.Row():
        gr.Markdown(
            """
            # 🏢 주택청약 스마트 AI 챗봇
            부동산 공무원과 청약 전문가의 지식을 학습한 AI입니다. 궁금한 점을 편하게 질문해 주세요!
            """
        )

    # 본문 영역 (좌/우 2단 분리)
    with gr.Row():

        #[좌측 사이드바] 이용 안내 및 팁 (화면의 1/3 차지)
        with gr.Column(scale=1):
            gr.Markdown("### 💡 챗봇 이용 가이드")
            gr.Info("질문은 최대 500자 이내로 구체적인 문장 형태로 적어주세요.")

            with gr.Accordion("📌 자주 묻는 질문 (클릭해서 복사하세요)", open=True):
                gr.Markdown(
                    """
                    - 주택청약종합저축이란 무엇인가요?
                    - 무주택 세대주 요건이 어떻게 되나요?
                    - 당첨자 발표 확인하고 서류 제출하는 법 알려줘
                    - 청약통장 가입 후 언제부터 1순위가 되나요?
                    """
                )

            with gr.Accordion("⚠️ 주의사항", open=False):
                gr.Markdown(
                    """
                    * 단답형(예: "청약")보다는 문장형(예: "청약 통장 가입 조건은?")이 더 정확합니다.
                    * AI의 답변은 참고용이며, 실제 법적 효력을 갖지 않습니다.
                    * 개인정보(주민번호, 계좌번호 등)는 입력하지 마세요.
                    """
                )

            gr.Image(
                "https://cdn-icons-png.flaticon.com/512/3135/3135692.png",
                width=100,
                show_label=False,
                show_download_button=False,
                interactive=False
            )

        # [우측 메인 화면] 실제 채팅창 (화면의 2/3 차지)
        with gr.Column(scale=2):
            # 봇과 유저의 프로필 이미지(아바타) 설정
            chatbot_ui = gr.Chatbot(
                height=500, # 채팅창 세로 길이
                avatar_images=(
                    "https://cdn-icons-png.flaticon.com/512/1077/1077114.png", # 유저 아이콘
                    "https://cdn-icons-png.flaticon.com/512/4712/4712027.png"  # AI 로봇 아이콘
                ),
                show_label=False
            )

            # 입력창 디자인 커스텀
            chat_input = gr.Textbox(
                placeholder="여기에 질문을 입력하고 Enter를 누르세요. (예: 무주택자 청약 신청 절차가 어떻게 되나요?)",
                container=False,
                scale=7
            )

            # ChatInterface를 Blocks 안에 삽입
            gr.ChatInterface(
                fn=chat_with_bot,
                chatbot=chatbot_ui,
                textbox=chat_input,
                examples=[
                    "주택청약종합저축이란 무엇인가요?",
                    "무주택 세대주 요건이 어떻게 되나요?",
                    "청약 당첨 이후 절차는 어떻게 진행되나요?"
                ],
                theme=custom_theme
            )

# 4. 앱 실행
if __name__ == "__main__":
    demo.launch(share=True)

/tmp/ipykernel_266/824544468.py:15: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, title="주택청약 AI 도우미") as demo:


질문은 최대 500자 이내로 구체적인 문장 형태로 적어주세요.


/tmp/ipykernel_266/824544468.py:64: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(
/tmp/ipykernel_266/824544468.py:64: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5fcae65837b70547f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Weekend 1 완료! 🎉

다음 주: 벡터 임베딩 + RAG 파이프라인